# 🚗 Estimation du Prix des Véhicules d'Occasion — Car Dekho
**Étudiant :** Meimoune Baba Cheikh Sidiya  
**Matricule :** C34620  
**Professeur :** Ezyn SEGNANE  
**Type de tâche :** Régression supervisée  
**Dataset :** Vehicle Dataset (CarDekho) — multi-villes fusionné


## 1. Imports et Configuration

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
import joblib

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("teal")
print("Imports OK ✅")


## 2. Chargement et Exploration des Données (EDA)

In [ ]:

# Chargement du dataset nettoyé (fusion multi-villes déjà effectuée)
df_raw = pd.read_csv('../data/processed/car_dekho_cleaned.csv')

# Correction des colonnes booléennes
bool_cols = ['fuel_Diesel','fuel_Electric','fuel_LPG','fuel_Petrol',
             'seller_type_Individual','seller_type_Trustmark Dealer','transmission_Manual']
for c in bool_cols:
    df_raw[c] = df_raw[c].astype(int)

print("Shape:", df_raw.shape)
print("\nColonnes:", df_raw.columns.tolist())
df_raw.head()


## Note sur le prétraitement

Le dataset `car_dekho_cleaned.csv` est déjà prétraité (fusion multi-villes, extraction de la marque, encodage, standardisation).

Les transformateurs sauvegardés (`scaler.pkl`, `label_encoder.pkl`, `columns.pkl`) sont dans `models/`.

Les fonctions de prétraitement sont dans `src/feature_engineering.py` et `src/data_preprocessing.py`.

## 4. Préparation des Données — Train/Test Split

In [ ]:

# Séparation features / target
feature_cols = [c for c in df_raw.columns if c != 'selling_price']
X = df_raw[feature_cols]
y = df_raw['selling_price']

print(f"Features: {X.shape} | Target: {y.shape}")
print(f"Features utilisées: {feature_cols}")

# Séparation stratifiée (bins du prix)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"\nTrain: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Test:  {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)")

# KFold pour validation croisée
kf = KFold(n_splits=5, shuffle=True, random_state=42)
print("\nKFold CV (k=5) configuré ✅")
